Import modules

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pl
import subprocess
import pathlib
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)
data_folder = pathlib.Path(os.path.join(basedir, os.pardir, "data", "perple_x_data"))

Set the lithology file basename.

In [ ]:
basename = "dike_25"

Run vertex (the first and longest Perple_X command that performs the free energy minimization)

In [ ]:
input = str(data_folder / basename)
stdout = open(output_folder / str('vertex_'+ basename + '.log'), 'w')
stderr = open(output_folder / str('vertex_'+ basename + '.err'), 'w')
subprocess.run(["vertex"], input=input, text=True, stdout=stdout, stderr=stderr)
stdout.close()
stderr.close()

Run werami to interpolate the free energy minimization.

We select the options:
* basename: project name uses the same input files from vertex
* 2: 2D grid
* 36: all phase &/or system properties (system properties here) - try a more compact output would work, e.g. 6 or 8 or 25
* 1: gives one System summary per node, 3 gives that plus all phases on lines
* n: y to include fluid in modal properties
* y: to change grid definition from values in .dat to something else
* 473 1673: T bounds (K)
* 1000 80000: P bounds (bar)
* 241 396: T, P nodes, designed for convenient/even 5C/0.02GPa grid
* 0: end


In [ ]:
input=str(data_folder / basename)+"""
2
36
1
n
y
473 1673
1000 80000
241 396
0
"""
stdout = open(output_folder / str('werami_'+ basename + '.log'), 'w')
stderr = open(output_folder / str('werami_'+ basename + '.err'), 'w')
subprocess.run(["werami"], input=input, text=True, stdout=stdout, stderr=stderr)
stdout.close()
stderr.close()

Set the output file name from the previous commands that we now wish to read to plot H$_2$O contents

In [ ]:
datafile = data_folder / str(basename + '_1.tab')

Look for the start of the data based on the column names

In [ ]:
cols = ["T(K)", "P(bar)", "H2O,wt%"]

# we need to find the row of the file that contains the header
header_idx = None
with open(datafile, 'r') as f:
    i = 0
    for line in f:
        if all([c in line for c in cols]):
            header_idx = i
            break
        i += 1

# some sanity checks
if header_idx is None:
    raise RuntimeError("Could not find header row")

if header_idx < 1:
    raise RuntimeError("Unexpected number of header rows")

Read the data part of the file

In [ ]:
df = pd.read_csv(datafile, sep=r"\s+", skiprows=header_idx-1, header=1, usecols=cols)
df

Get the data

In [ ]:
P = np.unique(df['P(bar)'].to_numpy())/10000.0
T = np.unique(df['T(K)'].to_numpy()) - 273.15
H2O = df['H2O,wt%'].to_numpy().reshape(len(P),len(T))

Plot the data

In [ ]:
fig, ax = pl.subplots(figsize=(7, 4.5))
vmin = 0.0
vmax = 5.5
dv = 0.25
levels = np.arange(vmin, vmax+dv, dv)
c = ax.contourf(T, P, H2O, levels=levels, cmap="jet_r")
cbar = fig.colorbar(c, label=r"H$_2$O (wt%)")
cbar.set_ticks(np.arange(vmin, vmax, 1, dtype=np.int32))
ax.set_ylabel(r"P (GPa)")
ax.set_xlabel(r"T ($^\circ$C)")
ax.set_box_aspect(1)